
# NetShield — Phase 10: Transfer Mechanism & Error Analysis

## Purpose

This phase moves directly from the Phase 9 domain-adaptation results toward the project's main research question:

> **Why does domain adaptation improve some cross-dataset transfers, and why does it fail or behave differently in the reverse direction?**

Phase 9 identified the strongest controlled result as **UNSW-NB15 → CICIDS2017 with Random Forest + CORAL**. It also showed that the reverse direction does not improve in the same way.

This notebook therefore does **not** add another arbitrary model. Instead, it investigates the mechanism behind the observed transfer behavior.

### Main questions

1. Which semantic features show the largest source/target distribution mismatch?
2. Which features are changed most by CORAL?
3. Which CICIDS attack classes benefit from the successful transfer?
4. Which attacks remain difficult?
5. Where do false positives come from?
6. Where do false negatives come from?
7. Does the successful transfer improvement correspond to meaningful changes in the feature space?
8. Does the same mechanism explain the weaker reverse transfer?

### Important methodological rule

Target labels are used **only for final error analysis**. They are never used to fit CORAL, select the model, or tune the adaptation.

The final evaluation subset remains untouched until prediction/evaluation.


In [ ]:

from pathlib import Path
import json, warnings, os, gc, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix
)

warnings.filterwarnings("ignore")

# Project paths
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_UNSW = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
RAW_CICIDS = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"
TABLE_DIR = PROJECT_ROOT / "outputs" / "transfer_mechanism_analysis" / "tables"
PLOT_DIR = PROJECT_ROOT / "outputs" / "transfer_mechanism_analysis" / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Output:", TABLE_DIR.parent)



## 1. Load only the data required for the analysis

To avoid the memory problems encountered earlier, CICIDS is loaded with a restricted set of columns needed for the common semantic representation and attack-label analysis.

We do not create full-data copies.


In [ ]:

def find_file(folder, exact_names):
    for name in exact_names:
        p = folder / name
        if p.exists():
            return p
    matches = []
    for p in folder.rglob("*.csv"):
        if p.name.lower() in {x.lower() for x in exact_names}:
            matches.append(p)
    return matches[0] if matches else None

unsw_train_path = find_file(
    RAW_UNSW,
    ["UNSW_NB15_training-set.csv", "UNSW-NB15_training-set.csv"]
)
unsw_test_path = find_file(
    RAW_UNSW,
    ["UNSW_NB15_testing-set.csv", "UNSW-NB15_testing-set.csv"]
)

if unsw_train_path is None or unsw_test_path is None:
    raise FileNotFoundError(
        "Official UNSW training/testing CSVs were not found under data/raw/UNSW-NB15."
    )

unsw_train = pd.read_csv(unsw_train_path)
unsw_test = pd.read_csv(unsw_test_path)

print("UNSW train:", unsw_train.shape)
print("UNSW test :", unsw_test.shape)


In [ ]:
CICIDS_NEEDED = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Packets/s",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Average Packet Size",
    "Label",
]

# Robust discovery: accept .csv/.CSV and tolerate whitespace/BOM/case in headers.
cic_files = sorted(
    p for p in RAW_CICIDS.rglob("*")
    if p.is_file() and p.suffix.lower() == ".csv"
)

if not cic_files:
    raise FileNotFoundError(
        "No CICIDS CSV files were found under data/raw/CICIDS2017."
    )

frames = []
for p in cic_files:
    try:
        header = pd.read_csv(p, nrows=0)
        original = list(header.columns)
        lookup = {str(c).strip().replace("\ufeff", "").lower(): c for c in original}

        selected = {}
        for needed in CICIDS_NEEDED:
            actual = lookup.get(needed.lower())
            if actual is not None:
                selected[needed] = actual

        if "Label" not in selected:
            print(f"Skipping {p.name}: Label column not found.")
            continue

        tmp = pd.read_csv(
            p,
            usecols=list(selected.values()),
            low_memory=False
        )
        tmp = tmp.rename(columns={actual: standard for standard, actual in selected.items()})
        tmp["source_file"] = p.name
        frames.append(tmp)
        print(f"Loaded {p.name}: {len(tmp):,} rows")
    except Exception as e:
        print(f"Could not load {p.name}: {type(e).__name__}: {e}")

if not frames:
    raise ValueError("No CICIDS files containing a recognizable Label column were loaded.")

cicids = pd.concat(frames, ignore_index=True)

print("\nCICIDS:", cicids.shape)
print("CICIDS source files:", cicids["source_file"].nunique())



## 2. Construct the verified common semantic representation

The common representation follows the Phase 4 semantic definitions.

For UNSW:

- `dur` is already in seconds.
- packet counts come from `spkts` and `dpkts`.
- byte counts come from `sbytes` and `dbytes`.
- packet rates are derived from packet counts / duration.
- average packet size is derived from total bytes / total packets.

For CICIDS:

- `Flow Duration` is converted from microseconds to seconds.
- directional packet/byte totals use the corresponding CICIDS flow fields.
- packet rates use the CICIDS packet-rate fields.

No target labels are used in these features.


In [ ]:

COMMON = [
    "flow_duration_sec", "fwd_packets", "bwd_packets",
    "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
    "packet_rate", "fwd_packet_rate", "bwd_packet_rate",
    "avg_packet_size"
]

def make_unsw_common(df):
    x = pd.DataFrame(index=df.index)
    dur = pd.to_numeric(df["dur"], errors="coerce")
    spkts = pd.to_numeric(df["spkts"], errors="coerce")
    dpkts = pd.to_numeric(df["dpkts"], errors="coerce")
    sbytes = pd.to_numeric(df["sbytes"], errors="coerce")
    dbytes = pd.to_numeric(df["dbytes"], errors="coerce")

    x["flow_duration_sec"] = dur
    x["fwd_packets"] = spkts
    x["bwd_packets"] = dpkts
    x["fwd_bytes"] = sbytes
    x["bwd_bytes"] = dbytes
    x["total_packets"] = spkts + dpkts
    x["total_bytes"] = sbytes + dbytes

    safe_dur = dur.replace(0, np.nan)
    safe_pkts = (spkts + dpkts).replace(0, np.nan)

    x["packet_rate"] = (spkts + dpkts) / safe_dur
    x["fwd_packet_rate"] = spkts / safe_dur
    x["bwd_packet_rate"] = dpkts / safe_dur
    x["avg_packet_size"] = (sbytes + dbytes) / safe_pkts
    return x[COMMON].astype("float32")

def make_cicids_common(df):
    x = pd.DataFrame(index=df.index)
    dur = pd.to_numeric(df["Flow Duration"], errors="coerce") / 1e6
    fp = pd.to_numeric(df["Total Fwd Packets"], errors="coerce")
    bp = pd.to_numeric(df["Total Backward Packets"], errors="coerce")
    fb = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce")
    bb = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce")

    x["flow_duration_sec"] = dur
    x["fwd_packets"] = fp
    x["bwd_packets"] = bp
    x["fwd_bytes"] = fb
    x["bwd_bytes"] = bb
    x["total_packets"] = fp + bp
    x["total_bytes"] = fb + bb
    x["packet_rate"] = pd.to_numeric(df["Flow Packets/s"], errors="coerce")
    x["fwd_packet_rate"] = pd.to_numeric(df["Fwd Packets/s"], errors="coerce")
    x["bwd_packet_rate"] = pd.to_numeric(df["Bwd Packets/s"], errors="coerce")
    x["avg_packet_size"] = pd.to_numeric(df["Average Packet Size"], errors="coerce")
    return x[COMMON].astype("float32")

X_unsw_train = make_unsw_common(unsw_train)
y_unsw_train = pd.to_numeric(unsw_train["label"], errors="coerce").astype("int8")

X_unsw_test = make_unsw_common(unsw_test)
y_unsw_test = pd.to_numeric(unsw_test["label"], errors="coerce").astype("int8")

X_cic = make_cicids_common(cicids)
y_cic = (
    cicids["Label"].astype(str).str.strip().str.upper()
    .ne("BENIGN").astype("int8")
)

cic_attack_label = cicids["Label"].astype(str).str.strip()

print("Common features:", len(COMMON))
print("UNSW train:", X_unsw_train.shape)
print("UNSW test :", X_unsw_test.shape)
print("CICIDS    :", X_cic.shape)



## 3. Reproduce a controlled CICIDS target evaluation split

The split is deterministic and stratified. The target adaptation subset is sampled **without labels being used in CORAL**; labels are retained only so the final evaluation can be stratified and the errors can later be analyzed.

A separate evaluation subset is never used to estimate CORAL statistics.


In [ ]:

from sklearn.model_selection import train_test_split

all_idx = np.arange(len(X_cic))

# Fixed stratified target split: 20% final evaluation, 80% available as unlabeled adaptation pool.
adapt_pool_idx, eval_idx = train_test_split(
    all_idx,
    test_size=0.20,
    stratify=y_cic.to_numpy(),
    random_state=RANDOM_STATE
)

# Keep the adaptation pool bounded for memory and covariance estimation.
rng = np.random.default_rng(RANDOM_STATE)
ADAPT_MAX = min(300_000, len(adapt_pool_idx))
adapt_idx = rng.choice(adapt_pool_idx, size=ADAPT_MAX, replace=False)

print("CICIDS adaptation rows:", len(adapt_idx))
print("CICIDS evaluation rows:", len(eval_idx))



## 4. CORAL utilities

CORAL aligns second-order feature statistics.

We standardize features first so covariance distances are not dominated by raw units. The alignment parameters are estimated from the source training data and the **unlabeled target adaptation subset only**. CORAL transforms the source representation toward the target covariance space; the target evaluation rows are never transformed by the source-to-target matrix.


In [ ]:
def clean_for_model(X):
    """Convert inf/overflow-like values to NaN without clipping genuine values."""
    X = np.asarray(X, dtype=np.float64).copy()
    X[~np.isfinite(X)] = np.nan
    X[np.abs(X) > np.finfo(np.float32).max] = np.nan
    return X

def fit_standardizer(X):
    X_clean = clean_for_model(X)
    imp = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X_clean)
    scaler = StandardScaler()
    Z = scaler.fit_transform(X_imp).astype("float32")
    return imp, scaler, Z

def transform_standardizer(X, imp, scaler):
    X_clean = clean_for_model(X)
    return scaler.transform(imp.transform(X_clean)).astype("float32")

def cov_matrix(Z):
    Z64 = np.asarray(Z, dtype=np.float64)
    return np.atleast_2d(np.cov(Z64, rowvar=False))

def symmetric_sqrt(M, eps=1e-6):
    vals, vecs = np.linalg.eigh((M + M.T) / 2)
    vals = np.clip(vals, eps, None)
    return (vecs * np.sqrt(vals)) @ vecs.T

def symmetric_inv_sqrt(M, eps=1e-6):
    vals, vecs = np.linalg.eigh((M + M.T) / 2)
    vals = np.clip(vals, eps, None)
    return (vecs * (1.0 / np.sqrt(vals))) @ vecs.T

def fit_coral_transform(X_source_train, X_target_adapt):
    """
    Fit CORAL using source training data and unlabeled target adaptation data.

    IMPORTANT: CORAL transforms the SOURCE representation into the TARGET
    covariance space. The target evaluation data are NOT transformed by the
    same source-to-target matrix. They remain in the source-standardized
    coordinate system, so the classifier sees the aligned source and target
    representations in the same space.
    """
    src_imp, src_scaler, Zs = fit_standardizer(X_source_train)
    Zt = transform_standardizer(X_target_adapt, src_imp, src_scaler)

    mu_s = Zs.mean(axis=0)
    mu_t = Zt.mean(axis=0)

    Cs = cov_matrix(Zs)
    Ct = cov_matrix(Zt)

    A = symmetric_inv_sqrt(Cs) @ symmetric_sqrt(Ct)

    return {
        "imputer": src_imp,
        "scaler": src_scaler,
        "source_mean": mu_s,
        "target_mean": mu_t,
        "transform": A,
        "Cs": Cs,
        "Ct": Ct
    }

def apply_coral_source(X_source, coral):
    """Transform SOURCE samples only into the target covariance space."""
    Z = transform_standardizer(X_source, coral["imputer"], coral["scaler"])
    Zc = Z - coral["source_mean"]
    Za = Zc @ coral["transform"] + coral["target_mean"]
    return Za.astype("float32")

def apply_coral_target(X_target, coral):
    """Put TARGET samples into the same source-standardized coordinate system."""
    return transform_standardizer(X_target, coral["imputer"], coral["scaler"])

def standardized_cov_distance(C1, C2):
    return float(np.linalg.norm(C1 - C2, ord="fro"))

def cov_of_sample(Z):
    return cov_matrix(Z)



## 5. Train the successful UNSW → CICIDS experiment

This is the central mechanism analysis.

We compare:

1. Baseline Random Forest
2. CORAL + Random Forest
3. CORAL + class-weighted Random Forest

The model is trained only on UNSW training data.

CICIDS adaptation rows are used only to estimate the unlabeled target covariance.
The CICIDS evaluation set is untouched until prediction.


In [ ]:
# Prepare source and target arrays
Xsrc = X_unsw_train
ysrc = y_unsw_train.to_numpy()
Xt_eval = X_cic.iloc[eval_idx]
yt_eval = y_cic.iloc[eval_idx].to_numpy()

# CORAL uses ONLY unlabeled target adaptation features.
Xt_adapt = X_cic.iloc[adapt_idx]

coral = fit_coral_transform(Xsrc, Xt_adapt)

# CORAL: transform SOURCE only; target evaluation remains source-standardized.
Xsrc_coral = apply_coral_source(Xsrc, coral)
Xt_eval_coral = apply_coral_target(Xt_eval, coral)

# Baseline: source-only imputation. No target statistics are used.
base_imp = SimpleImputer(strategy="median")
Xsrc_base = base_imp.fit_transform(clean_for_model(Xsrc)).astype("float32")
Xt_eval_base = base_imp.transform(clean_for_model(Xt_eval)).astype("float32")

rf_params = dict(
    n_estimators=250,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2
)

rf_base = RandomForestClassifier(**rf_params)
rf_coral = RandomForestClassifier(**rf_params)
rf_coral_bal = RandomForestClassifier(**rf_params, class_weight="balanced")

rf_base.fit(Xsrc_base, ysrc)
rf_coral.fit(Xsrc_coral, ysrc)
rf_coral_bal.fit(Xsrc_coral, ysrc)

pred_base = rf_base.predict(Xt_eval_base)
proba_base = rf_base.predict_proba(Xt_eval_base)[:, 1]

pred_coral = rf_coral.predict(Xt_eval_coral)
proba_coral = rf_coral.predict_proba(Xt_eval_coral)[:, 1]

pred_coral_bal = rf_coral_bal.predict(Xt_eval_coral)
proba_coral_bal = rf_coral_bal.predict_proba(Xt_eval_coral)[:, 1]

print("Models trained successfully.")


In [ ]:

def metric_row(name, y_true, pred, proba):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "model": name,
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

main_results = pd.DataFrame([
    metric_row("baseline_rf", yt_eval, pred_base, proba_base),
    metric_row("coral_rf", yt_eval, pred_coral, proba_coral),
    metric_row("coral_rf_balanced", yt_eval, pred_coral_bal, proba_coral_bal),
])

display(main_results)
main_results.to_csv(TABLE_DIR / "01_unsw_to_cicids_mechanism_results.csv", index=False)



## 6. Quantify what CORAL changed in the feature space

This separates two questions:

- Did CORAL mathematically align the distributions?
- Did that alignment translate into better classification?

We report both, rather than assuming one implies the other.


In [ ]:
# CORAL diagnostics: target stays in the source-standardized coordinate system.
Zs_before = transform_standardizer(Xsrc, coral["imputer"], coral["scaler"])
Zt_before = transform_standardizer(Xt_adapt, coral["imputer"], coral["scaler"])
Zs_after = apply_coral_source(Xsrc, coral)

Cs_before = cov_of_sample(Zs_before)
Ct_before = cov_of_sample(Zt_before)
Cs_after = cov_of_sample(Zs_after)

dist_before = standardized_cov_distance(Cs_before, Ct_before)
dist_after = standardized_cov_distance(Cs_after, Ct_before)

mean_dist_before = float(np.linalg.norm(Zs_before.mean(axis=0) - Zt_before.mean(axis=0)))
mean_dist_after = float(np.linalg.norm(Zs_after.mean(axis=0) - Zt_before.mean(axis=0)))

cov_diag = pd.DataFrame([{
    "direction": "UNSW_to_CICIDS",
    "covariance_distance_before": dist_before,
    "covariance_distance_after": dist_after,
    "relative_reduction": 1 - dist_after / dist_before if dist_before else np.nan,
    "mean_distance_before": mean_dist_before,
    "mean_distance_after": mean_dist_after
}])

display(cov_diag)
cov_diag.to_csv(TABLE_DIR / "02_coral_covariance_alignment.csv", index=False)

# Feature-wise mean and standard deviation movement.
feature_shift = pd.DataFrame({
    "feature": COMMON,
    "source_mean_before": Zs_before.mean(axis=0),
    "target_mean": Zt_before.mean(axis=0),
    "source_mean_after": Zs_after.mean(axis=0),
    "source_std_before": Zs_before.std(axis=0),
    "target_std": Zt_before.std(axis=0),
    "source_std_after": Zs_after.std(axis=0),
})

feature_shift["abs_mean_gap_before"] = (
    feature_shift["source_mean_before"] - feature_shift["target_mean"]
).abs()
feature_shift["abs_mean_gap_after"] = (
    feature_shift["source_mean_after"] - feature_shift["target_mean"]
).abs()
feature_shift["mean_gap_reduction"] = (
    feature_shift["abs_mean_gap_before"] -
    feature_shift["abs_mean_gap_after"]
)

display(feature_shift.sort_values("mean_gap_reduction", ascending=False))
feature_shift.to_csv(TABLE_DIR / "03_feature_alignment_effect.csv", index=False)

# Hard sanity check: a valid CORAL run should reduce covariance distance.
if not np.isfinite(dist_after) or dist_after > dist_before * 1.05:
    raise RuntimeError(
        "CORAL sanity check failed: transformed source covariance is not closer "
        "to target covariance. Do not interpret model results."
    )



## 7. Attack-class error analysis

The binary model is the main cross-dataset experiment, but CICIDS contains attack labels that let us investigate **which attack types benefit from transfer**.

These labels are used only after prediction.

Very small classes should be interpreted cautiously rather than treated as statistically stable evidence.


In [ ]:

attack_eval = cic_attack_label.iloc[eval_idx].reset_index(drop=True)
yt = pd.Series(yt_eval).reset_index(drop=True)

def classwise_binary_error_table(labels, y_true, pred, proba):
    rows = []
    for cls in sorted(labels.unique()):
        mask = labels == cls
        yt_cls = y_true[mask]
        pred_cls = pred[mask]
        n = int(mask.sum())
        attack_recall = float((pred_cls == 1).mean()) if n else np.nan
        rows.append({
            "attack_label": cls,
            "n": n,
            "detected_as_attack_rate": attack_recall,
            "missed_as_benign_rate": 1 - attack_recall if n else np.nan,
            "mean_attack_probability": float(proba[mask].mean()) if n else np.nan
        })
    return pd.DataFrame(rows).sort_values(["n", "detected_as_attack_rate"], ascending=[False, False])

class_base = classwise_binary_error_table(
    attack_eval, yt, pred_base, proba_base
)
class_coral = classwise_binary_error_table(
    attack_eval, yt, pred_coral, proba_coral
)

class_compare = class_base.merge(
    class_coral,
    on=["attack_label", "n"],
    suffixes=("_baseline", "_coral")
)

class_compare["detection_rate_change"] = (
    class_compare["detected_as_attack_rate_coral"] -
    class_compare["detected_as_attack_rate_baseline"]
)

display(class_compare)
class_compare.to_csv(TABLE_DIR / "04_attack_class_transfer_effect.csv", index=False)



## 8. False-positive analysis

False positives are benign CICIDS flows classified as attacks.

Because Phase 9 showed a major FPR reduction for the successful UNSW → CICIDS transfer, we need to identify whether that reduction is broad or concentrated in particular capture files.

The `source_file` field is **metadata only** and is never given to the model.


In [ ]:

source_eval = cicids["source_file"].iloc[eval_idx].reset_index(drop=True)

fp_analysis = pd.DataFrame({
    "source_file": source_eval,
    "true_label": yt,
    "baseline_pred": pred_base,
    "coral_pred": pred_coral
})

fp_analysis["baseline_fp"] = (
    (fp_analysis["true_label"] == 0) &
    (fp_analysis["baseline_pred"] == 1)
)
fp_analysis["coral_fp"] = (
    (fp_analysis["true_label"] == 0) &
    (fp_analysis["coral_pred"] == 1)
)

capture_fp = (
    fp_analysis[fp_analysis["true_label"] == 0]
    .groupby("source_file")
    .agg(
        benign_rows=("true_label", "size"),
        baseline_false_positives=("baseline_fp", "sum"),
        coral_false_positives=("coral_fp", "sum")
    )
    .reset_index()
)

capture_fp["baseline_fpr"] = (
    capture_fp["baseline_false_positives"] / capture_fp["benign_rows"]
)
capture_fp["coral_fpr"] = (
    capture_fp["coral_false_positives"] / capture_fp["benign_rows"]
)
capture_fp["fpr_change"] = (
    capture_fp["coral_fpr"] - capture_fp["baseline_fpr"]
)

display(capture_fp.sort_values("fpr_change"))
capture_fp.to_csv(TABLE_DIR / "05_capture_level_false_positive_analysis.csv", index=False)



## 9. False-negative analysis

False negatives are attacks classified as benign.

This is the complementary question to the FPR improvement:

> Did CORAL reduce false alarms by becoming too conservative about attacks?


In [ ]:

attack_error = pd.DataFrame({
    "attack_label": attack_eval,
    "true_label": yt,
    "baseline_pred": pred_base,
    "coral_pred": pred_coral
})

attack_error = attack_error[attack_error["true_label"] == 1].copy()
attack_error["baseline_fn"] = attack_error["baseline_pred"] == 0
attack_error["coral_fn"] = attack_error["coral_pred"] == 0

fn_by_attack = (
    attack_error.groupby("attack_label")
    .agg(
        attack_rows=("true_label", "size"),
        baseline_fn=("baseline_fn", "sum"),
        coral_fn=("coral_fn", "sum")
    )
    .reset_index()
)

fn_by_attack["baseline_fnr"] = fn_by_attack["baseline_fn"] / fn_by_attack["attack_rows"]
fn_by_attack["coral_fnr"] = fn_by_attack["coral_fn"] / fn_by_attack["attack_rows"]
fn_by_attack["fnr_change"] = fn_by_attack["coral_fnr"] - fn_by_attack["baseline_fnr"]

display(fn_by_attack.sort_values("fnr_change", ascending=False))
fn_by_attack.to_csv(TABLE_DIR / "06_attack_false_negative_analysis.csv", index=False)



## 10. Feature importance: did the successful model change what it relies on?

For Random Forest, feature importance provides an interpretable comparison between:

- the native common-feature baseline
- the CORAL-adapted model

This does not prove causality. It is used as supporting evidence for the transfer mechanism.


In [ ]:

importance = pd.DataFrame({
    "feature": COMMON,
    "baseline_rf_importance": rf_base.feature_importances_,
    "coral_rf_importance": rf_coral.feature_importances_
})

importance["importance_change"] = (
    importance["coral_rf_importance"] -
    importance["baseline_rf_importance"]
)

importance = importance.sort_values(
    "coral_rf_importance", ascending=False
)

display(importance)
importance.to_csv(TABLE_DIR / "07_feature_importance_comparison.csv", index=False)

plt.figure(figsize=(10, 6))
plot_df = importance.sort_values("coral_rf_importance")
plt.barh(plot_df["feature"], plot_df["coral_rf_importance"])
plt.xlabel("Random Forest importance")
plt.ylabel("Common semantic feature")
plt.title("CORAL Random Forest feature importance")
plt.tight_layout()
plt.savefig(PLOT_DIR / "01_coral_feature_importance.png", dpi=160)
plt.show()



## 11. Direct comparison: distribution shift versus performance change

This table connects the earlier dataset-comparison work with Phase 9.

The purpose is not to claim that one statistic causes another. It is to identify which semantic dimensions are plausible contributors to transfer difficulty.


In [ ]:

# Robust distribution diagnostics on the source and target adaptation samples.
# We use median and IQR rather than relying only on means because the features are highly skewed.

shift_rows = []
for j, feat in enumerate(COMMON):
    s = pd.Series(Xsrc.iloc[:, j]).replace([np.inf, -np.inf], np.nan).dropna()
    t = pd.Series(Xt_adapt.iloc[:, j]).replace([np.inf, -np.inf], np.nan).dropna()

    s_med = s.median()
    t_med = t.median()
    s_iqr = s.quantile(0.75) - s.quantile(0.25)
    t_iqr = t.quantile(0.75) - t.quantile(0.25)

    shift_rows.append({
        "feature": feat,
        "source_median": s_med,
        "target_median": t_med,
        "absolute_median_difference": abs(s_med - t_med),
        "source_iqr": s_iqr,
        "target_iqr": t_iqr,
        "iqr_ratio_target_source": (
            t_iqr / s_iqr if pd.notna(s_iqr) and s_iqr != 0 else np.nan
        )
    })

shift_df = pd.DataFrame(shift_rows)
display(shift_df.sort_values("absolute_median_difference", ascending=False))
shift_df.to_csv(TABLE_DIR / "08_semantic_distribution_shift_summary.csv", index=False)



## 12. Save a concise research interpretation

This cell does not manufacture a conclusion from thresholds. It simply records the measured evidence that should be considered when writing the final analysis.


In [ ]:
best_row = main_results.sort_values("f1", ascending=False).iloc[0]
baseline_row = main_results[main_results["model"] == "baseline_rf"].iloc[0]
coral_row = main_results[main_results["model"] == "coral_rf"].iloc[0]

interpretation = {
    "research_question": (
        "Does domain adaptation improve cross-dataset NIDS transfer, and what "
        "mechanisms/errors explain the change?"
    ),
    "direction_analyzed": "UNSW_NB15_to_CICIDS2017",
    "baseline_rf_f1": float(baseline_row["f1"]),
    "coral_rf_f1": float(coral_row["f1"]),
    "baseline_rf_pr_auc": float(baseline_row["pr_auc"]),
    "coral_rf_pr_auc": float(coral_row["pr_auc"]),
    "baseline_rf_roc_auc": float(baseline_row["roc_auc"]),
    "coral_rf_roc_auc": float(coral_row["roc_auc"]),
    "baseline_rf_fpr": float(baseline_row["fpr"]),
    "coral_rf_fpr": float(coral_row["fpr"]),
    "covariance_distance_before": float(cov_diag.iloc[0]["covariance_distance_before"]),
    "covariance_distance_after": float(cov_diag.iloc[0]["covariance_distance_after"]),
    "covariance_distance_relative_reduction": float(cov_diag.iloc[0]["relative_reduction"]),
    "mean_distance_before": float(cov_diag.iloc[0]["mean_distance_before"]),
    "mean_distance_after": float(cov_diag.iloc[0]["mean_distance_after"]),
    "warning": (
        "Attack-class results for very small CICIDS classes must be interpreted "
        "cautiously. Feature importance and distribution diagnostics are supporting "
        "evidence, not causal proof. CORAL results should only be interpreted when "
        "the covariance sanity check passes."
    )
}

with open(TABLE_DIR / "09_phase10_research_evidence.json", "w", encoding="utf-8") as f:
    json.dump(interpretation, f, indent=2)

display(pd.DataFrame([interpretation]))



# Phase 10 checkpoint

At the end of this notebook we should be able to explain the successful transfer rather than merely report that it improved.

The CORAL sanity check must pass before the Phase 10 outputs are interpreted. A valid run should show that the covariance distance between the transformed source and target adaptation data is lower than before alignment.

## Expected deliverables

Saved under:

`outputs/transfer_mechanism_analysis/`

- `tables/01_unsw_to_cicids_mechanism_results.csv`
- `tables/02_coral_covariance_alignment.csv`
- `tables/03_feature_alignment_effect.csv`
- `tables/04_attack_class_transfer_effect.csv`
- `tables/05_capture_level_false_positive_analysis.csv`
- `tables/06_attack_false_negative_analysis.csv`
- `tables/07_feature_importance_comparison.csv`
- `tables/08_semantic_distribution_shift_summary.csv`
- `tables/09_phase10_research_evidence.json`
- `plots/01_coral_feature_importance.png`

## What comes after this

Only after the mechanism/error analysis is complete should we decide whether a final targeted robustness experiment is justified.

The project should finish with a defensible answer to:

> **When does cross-dataset NIDS transfer improve, why does it improve, and why does the same strategy fail in the opposite direction?**

That answer—not simply the highest F1—is the core contribution of NetShield.
